# Week 8 - Notebook 2: Test Individual Agents

## Goal
Test each agent individually:
1. SpecialistAgent (fine-tuned Llama on Modal)
2. FrontierAgent (GPT-5.1 + RAG)
3. Compare results

## Time: 15-20 minutes

In [ ]:
import sys
sys.path.append('..')

import os
from dotenv import load_dotenv
import chromadb

from src.agents import SpecialistAgent, FrontierAgent
from src.utils.items import Item
from src.utils.evaluator import evaluate
from src.config import config

# Load environment
load_dotenv()

print("✅ Environment loaded")

## Load Test Data

In [ ]:
print(f"Loading test data from: {config.DATASET_NAME}")
_, _, test = Item.from_hub(config.DATASET_NAME)

print(f"✅ Loaded {len(test):,} test items")

## Load ChromaDB Collection

In [ ]:
# Load ChromaDB
chroma_client = chromadb.PersistentClient(path="../data/chroma")
collection = chroma_client.get_collection(name="products")

print(f"✅ Loaded ChromaDB collection")
print(f"   Items in collection: {collection.count():,}")

## Test 1: SpecialistAgent (Fine-tuned Llama)

**Note:** This requires Modal.com setup from Week 7

In [ ]:
# Initialize SpecialistAgent
specialist = SpecialistAgent()

print("✅ SpecialistAgent initialized")

In [ ]:
# Test on a few examples
print("Testing SpecialistAgent on 3 products:\n")

for i in range(3):
    item = test[i]
    description = item.prompt or item.summary or item.title
    prediction = specialist.price(description)
    
    print(f"Product: {item.title[:50]}...")
    print(f"Actual: ${item.price:.2f}")
    print(f"Predicted: ${prediction:.2f}")
    print(f"Error: ${abs(prediction - item.price):.2f}\n")

In [ ]:
# Full evaluation
def specialist_predict(item):
    description = item.prompt or item.summary or item.title
    return specialist.price(description)

print("Running full evaluation on SpecialistAgent...\n")
evaluate(specialist_predict, test, size=200, workers=1)

## Test 2: FrontierAgent (GPT-5.1 + RAG)

In [ ]:
# Initialize FrontierAgent
frontier = FrontierAgent(collection)

print("✅ FrontierAgent initialized")

In [ ]:
# Test on a few examples
print("Testing FrontierAgent on 3 products:\n")

for i in range(3):
    item = test[i]
    description = item.prompt or item.summary or item.title
    prediction = frontier.price(description)
    
    print(f"Product: {item.title[:50]}...")
    print(f"Actual: ${item.price:.2f}")
    print(f"Predicted: ${prediction:.2f}")
    print(f"Error: ${abs(prediction - item.price):.2f}\n")

In [ ]:
# Full evaluation
def frontier_predict(item):
    description = item.prompt or item.summary or item.title
    return frontier.price(description)

print("Running full evaluation on FrontierAgent...\n")
evaluate(frontier_predict, test, size=200, workers=1)

## Test 3: Confidence Scores (NEW!)

In [ ]:
# Test confidence-aware predictions
print("Testing confidence scores on 5 products:\n")

for i in range(5):
    item = test[i]
    description = item.prompt or item.summary or item.title
    
    # Get predictions with confidence
    specialist_price, specialist_conf = specialist.price_with_confidence(description)
    frontier_price, frontier_conf = frontier.price_with_confidence(description)
    
    print(f"Product: {item.title[:50]}...")
    print(f"Actual: ${item.price:.2f}")
    print(f"\nSpecialist: ${specialist_price:.2f} (confidence: {specialist_conf:.2f})")
    print(f"Frontier: ${frontier_price:.2f} (confidence: {frontier_conf:.2f})")
    print("-" * 60 + "\n")

## Summary

✅ Individual agent testing complete!

**Expected Results:**
- SpecialistAgent: ~$40 error (fine-tuned Llama)
- FrontierAgent: ~$30 error (GPT-5.1 + RAG)

**Key Observations:**
1. FrontierAgent is more accurate (RAG helps!)
2. Confidence scores vary by product
3. Some products are easier to predict than others

**Next step:** `03_ensemble.ipynb` - Combine agents with confidence weighting